<a href="https://colab.research.google.com/github/wingated/cs473/blob/main/labs/cs473_lab_week_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b>After clicking the "Open in Colab" link, copy the notebook to your own Google Drive before getting started, or it will not save your work</b></p>

# BYU CS 473 Lab Week 6

## Introduction:

The concept of mutual information is powerful in theory, but in practice it can be difficult to estimate. In this lab, you will implement a conceptually simple way to estimate the mutual information between two random variables. It is known as the KSG estimator (from "Kraskov–Stögbauer–Grassberger").

You will then use your estimator as part of a feature selection algorithm.

---
## Grading standards   

Your notebook will be graded on the following:

* 60% Correct implementation of KSG estimator
* 20% Correct implementation of "all_mutual_inf"
* 20% Correct selection of most informative columns

---
## Description

For this lab, we will use this dataset [lab_week_6_data.csv](https://github.com/wingated/cs473/blob/main/labs/data/lab_week_6_data.csv).

Either: download the csv from the link above, then upload the csv to your current google colab runtime,
or run the curl command below to fetch it directly into your runtime.

Then begin by loading and preparing the dataset:

In [1]:
!curl -L https://raw.githubusercontent.com/wingated/cs473/refs/heads/main/labs/data/lab_week_6_data.csv > lab_week_6_data.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 14967  100 14967    0     0  42738      0 --:--:-- --:--:-- --:--:-- 42641


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv( "lab_week_6_data.csv" )

X = df[['x1','x2','x3','x4','x5']]
y = df['y']

X = np.array(X)
y = np.array(y)


The KSG estimator is an elegant idea that's fairly easy to implement. The idea is to use an adaptive grid, where the grid depends on the local density of the data.  This is accomplished by calculating the k'th nearest neighbor for each point, and then defining a box around each point that just barely extends to include the k'th nearest neighbor. Importantly, the size of the box is also used in the marginal entropy calculations

To get some intuition for the method, check out [this demonstration on Wolfram Alpha's site](https://demonstrations.wolfram.com/KraskovKSGEstimatorOfMutualInformation/).  Here is a screenshot of their UI:

![Wolfram UI](https://raw.githubusercontent.com/wingated/cs473/main/labs/images/lab_week_6_image1.png)

The algorithm is fairly straightforward.  We are given N samples from the joint distribution of p(X,Y). First, we calculate the *local mutual information* for each data point:

1. For each datapoint $d_j = (x_j,y_j)$
2. Find the k'th nearest neighbor (call this point $n_j$) -- you should use the infinity norm (max norm), and distance should be calculated using both coordinates (ie, don't just calculate distances using only the x or y coordinate)
3. Count the number of points in x whose 'x-distance' fall within the distance to the k'th nearest neighbor (calculated above) of $d_j$ (ie, ignore y); call this number $n_{x{_j}}$.  Do the same for the points in y (these are the shaded gray boxes in the image); call it $n_{y{_j}}$.

Then, we calculate the final mutual information by averaging over all of the datapoints and adding a few more digamma calls:

![final mutual information](https://raw.githubusercontent.com/wingated/cs473/main/labs/images/lab_week_6_image3.png)

where $\Psi$ is the digamma function

If you want more details on the implementation, see page 2 of [Estimating Mutual Information](https://arxiv.org/pdf/cond-mat/0305641)

In [2]:
# This could be useful - scipy.special.digamma

def mutual_inf( X, Y, k ):
    # X and Y are lists of length n of samples from p(X,Y)
    # X is a list of x coordinates, Y is the y coordinates, so a point n is (X[n], Y[n])
    #
    # this function returns a single scalar
    # Since this is an estimation, occasionally you'll get negative values.
    # Make sure to clamp the return above zero!

    # your code here
    X, Y = np.asarray(X), np.asarray(Y)
    n = len(X)
    total = 0
    for j in range(n):
        X_diffs = np.abs(X - X[j])
        Y_diffs = np.abs(Y - Y[j])

        joint_diffs = np.maximum(X_diffs, Y_diffs)

        # Remove/ignore distance to itself.
        joint_diffs[j] = np.inf

        # Determine epsilon (kth nearest JOINT distance).
        epsilon = np.sort(joint_diffs)[k - 1]

        # Now count points within epsilon in each marginal.
        nx = np.sum(X_diffs < epsilon) - 1
        ny = np.sum(Y_diffs < epsilon) - 1

        # Use nx and ny in digamma expreession.
        total += special.digamma(nx + 1) + special.digamma(ny + 1)

    scalar = special.digamma(k) + special.digamma(n) -total / n

    return max(0, scalar)

---
## Part 2:

Now that you can estimate mutual information between two variables, we're going to use your method in the context of a regression problem.

Given our X,y data from the beginning of the lab, run your function and calculate the mutual information between every column of X and y.  This should result in a list of scores, that might look something like this:

```python
array([1.56167519, 0.23807644, 0.18881414, 0.17276823, 0.])
```

If you wrap this in a single function (call it "all_mutual_inf"), then you can use scikit learn's feature selection algorithms:

In [ ]:
def all_mutual_inf( X, y ):
    # return a list of mutual informations
    # the list should be as long as the number of columns in X

    # your code here
    scores = []

    for j in range(x.shape[1]):
        score = mutual_inf(x[:, j], y, ...)
        scores.append(score)

    return np.array(scores)

In [3]:
from sklearn.feature_selection import SelectKBest

Selector = SelectKBest(all_mutual_inf, k=3)
Selector.fit(X, y)
Selector.scores_ # Returns the information per column

NameError: name 'all_mutual_inf' is not defined

So: which X features are most informative about y?

---
## Hints

The following functions may be useful to you:

In [ ]:
scipy.special.digamma

For fun, you could also [read more about the KSG estimator](https://arxiv.org/pdf/1604.03006).